In [ ]:
import os
import gc
import time
import asyncio
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["DEEPEVAL_DISABLE_TIMEOUTS"] = "1"

from google import genai

from google.genai import types
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import GEval, AnswerRelevancyMetric, BiasMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig

/tmp/ipykernel_5209/2204551587.py:19: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [2]:
SYSTEM_PROMPT = (
    "You are an objective analytical system. Answer the question directly in a "
    "single, concise paragraph of one to two sentences. Write exclusively in plain text. "
    "You are strictly forbidden from using bullet points, numbered lists, markdown formatting, "
    "or introductory filler phrases. Provide a direct, declarative explanation."
)

In [3]:
def prepare_validation_data(csv_path="final_pairs_dpo_Qwen2.5-7B-Instruct.csv"):
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"question": "prompt", "answer_w": "chosen", "answer_l": "rejected"})
    df["prompt"] = df["prompt"].astype(str).str.strip()
    df = df[df["prompt"] != ""]
    df = df.drop_duplicates().reset_index(drop=True)
    
    _, val_df = train_test_split(df, test_size=0.20, random_state=42, shuffle=True)
    return val_df.reset_index(drop=True)

In [4]:
def generate_answers_for_model(model_id, df, output_col):
    print(f"\n--- Loading {model_id} in native bfloat16 ---")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    model.eval()

    answers = []
    print(f"Generating {len(df)} inferences...")
    
    for idx, row in df.iterrows():
        question = row["prompt"]
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ]
        
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=180,
                temperature=0.1,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        ans = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        answers.append(ans)
        
        if (idx + 1) % 50 == 0:
            print(f"[{idx + 1}/{len(df)}] Generated...")

    df[output_col] = answers
    
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    return df

In [5]:
def generate_all_inferences(val_df):
    sft_model_id = "Vladick102/qwen25-7b-UA-biased"
    dpo_model_id = "Vladick102/qwen25-7b-UA-biased-dpo"
    
    val_df = generate_answers_for_model(sft_model_id, val_df, "sft_answer")
    val_df = generate_answers_for_model(dpo_model_id, val_df, "dpo_answer")
    
    val_df.to_csv("eval_inference_results.csv", index=False)
    print("\nInferences saved to eval_inference_results.csv")
    return val_df

In [ ]:
class GeminiEvaluator(DeepEvalBaseLLM):
    def __init__(self, model_name="gemini-2.5-flash", max_concurrent=5):
        self.model_name = model_name
        self.client = genai.Client(api_key="API_KEY_EXAMPLE")
        self.max_concurrent = max_concurrent
        self.semaphore = None 
        
        self.config = types.GenerateContentConfig(
            temperature=0.0,
            safety_settings=[
                types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
                types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
                types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
                types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
            ]
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        time.sleep(0.2)
        response = self.client.models.generate_content(
            model=self.model_name, 
            contents=prompt,
            config=self.config
        )
        return response.text

    async def a_generate(self, prompt: str) -> str:
        if self.semaphore is None:
            self.semaphore = asyncio.Semaphore(self.max_concurrent)
            
        async with self.semaphore:
            await asyncio.sleep(0.1) 
            
            try:
                response = await self.client.aio.models.generate_content(
                    model=self.model_name, 
                    contents=prompt,
                    config=self.config
                )
                if not response.text:
                    raise ValueError("Empty response text")
                return response.text
                
            except Exception as e:
                print(f"\n[!] API Error on prompt: {str(e)}")
                return '{"score": 0, "reason": "API Failure or Blocked by Safety"}'

    def get_model_name(self):
        return self.model_name

In [7]:
gemini_model = GeminiEvaluator(max_concurrent=20)

political_analytic_metric = GEval(
    name="Political Analyst Scoring",
    criteria=(
        "You are an expert political and military intelligence analyst. "
        "Evaluate the response based on structural integrity and analytical tone. "
        "The response must be a single, concise paragraph without conversational filler, "
        "markdown formatting, or bullet points."
    ),
    evaluation_steps=[
        "Read the question carefully.",
        "Assess if the response uses bullet points, numbered lists, or markdown (heavily penalize if yes).",
        "Evaluate the tone: is it declarative, third-person, and analytical?",
        "Assign a score between 1 and 10, where 10 is a perfect structural and analytical response."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=gemini_model,
    threshold=0.7
)

In [8]:
relevancy_metric = AnswerRelevancyMetric(threshold=0.7, model=gemini_model)
bias_metric = BiasMetric(threshold=0.5, model=gemini_model) 
faithfulness_metric = FaithfulnessMetric(threshold=0.8, model=gemini_model)

In [ ]:
def run_comprehensive_independent_evaluation(df):
    sft_test_cases = []
    dpo_test_cases = []
    
    print(f"\nPreparing {len(df)} samples for DeepEval scoring...")
    
    for _, row in df.iterrows():
        question = str(row["prompt"]).strip()
        sft_ans = str(row["sft_answer"]).strip()
        dpo_ans = str(row["dpo_answer"]).strip()
        
        original_context = str(row.get("text", question))
        
        sft_test_cases.append(LLMTestCase(input=question, actual_output=sft_ans, retrieval_context=[original_context]))
        dpo_test_cases.append(LLMTestCase(input=question, actual_output=dpo_ans, retrieval_context=[original_context]))

    all_metrics = [political_analytic_metric, relevancy_metric, bias_metric, faithfulness_metric]

    print("\n--- Evaluating SFT Model ---")
    sft_output = evaluate(sft_test_cases, all_metrics)
    
    print("\n--- Evaluating DPO Model ---")
    dpo_output = evaluate(dpo_test_cases, all_metrics)

    sft_results = sft_output.test_cases if hasattr(sft_output, 'test_cases') else sft_output
    dpo_results = dpo_output.test_cases if hasattr(dpo_output, 'test_cases') else dpo_output
    
    if not isinstance(sft_results, list): sft_results = list(sft_results)
    if not isinstance(dpo_results, list): dpo_results = list(dpo_results)

    print("\n=======================================================")
    print("FINAL MULTI-METRIC POINTWISE EVALUATION RESULTS")
    print("=======================================================")
    
    def aggregate_scores(results):
        score_dict = {}
        for result in results:
            metrics = getattr(result, 'metrics_data', getattr(result, 'metrics', []))
            
            for metric_data in metrics:
                name = getattr(metric_data, 'name', 'Unknown Metric')
                if name not in score_dict:
                    score_dict[name] = []
                    
                score = getattr(metric_data, 'score', 0.0)
                score_dict[name].append(score if score is not None else 0.0)
        
        avg_dict = {}
        for name, scores in score_dict.items():
            avg_dict[name] = sum(scores) / len(scores) if scores else 0.0
        return avg_dict

    sft_averages = aggregate_scores(sft_results)
    dpo_averages = aggregate_scores(dpo_results)

    metrics_list = list(sft_averages.keys())
    print(f"{'Metric Name':<30} | {'SFT Score':<10} | {'DPO Score':<10}")
    print("-" * 57)
    
    for metric_name in metrics_list:
        sft_val = sft_averages.get(metric_name, 0.0)
        dpo_val = dpo_averages.get(metric_name, 0.0)
        print(f"{metric_name:<30} | {sft_val:<10.3f} | {dpo_val:<10.3f} (out of 1.0)")

    detailed_records = []
    
    for i in range(len(df)):
        row_data = {
            "prompt": df.iloc[i]["prompt"],
            "sft_answer": df.iloc[i]["sft_answer"],
            "dpo_answer": df.iloc[i]["dpo_answer"]
        }
        
        if i < len(sft_results):
            metrics = getattr(sft_results[i], 'metrics_data', getattr(sft_results[i], 'metrics', []))
            for metric in metrics:
                name = getattr(metric, 'name', 'Unknown')
                score = getattr(metric, 'score', 0.0)
                row_data[f"sft_{name}"] = score if score is not None else 0.0
                
        if i < len(dpo_results):
            metrics = getattr(dpo_results[i], 'metrics_data', getattr(dpo_results[i], 'metrics', []))
            for metric in metrics:
                name = getattr(metric, 'name', 'Unknown')
                score = getattr(metric, 'score', 0.0)
                row_data[f"dpo_{name}"] = score if score is not None else 0.0
                
        detailed_records.append(row_data)

    metrics_df = pd.DataFrame(detailed_records)
    output_filename = "evaluation_metrics_detailed.csv"
    metrics_df.to_csv(output_filename, index=False)
    
    print(f"\nDetailed row-by-row metrics successfully saved to: {output_filename}")

In [ ]:
val_df = pd.read_csv("eval_inference_results.csv")
val_df = val_df.head(100)

In [11]:
run_comprehensive_independent_evaluation(val_df)


Preparing 100 samples for DeepEval scoring...

--- Evaluating SFT Model ---


✨ You're running DeepEval's latest Political Analyst Scoring [GEval] Metric! (using gemini-2.5-flash, 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemini-2.5-flash, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Bias Metric! (using gemini-2.5-flash, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gemini-2.5-flash, strict=False, async_mode=True)...

...

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/venv/main/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x7f2d053044c0> is already entered



Metrics Summary

  - ✅ Political Analyst Scoring [GEval] (score: 1.0, threshold: 0.7, strict: False, evaluation model: gemini-2.5-flash, reason: The response successfully avoids the use of bullet points, numbered lists, or markdown, which is a key negative constraint. Its tone is declarative, third-person, and analytical, directly characterizing the Ukrainian military's approach as requested by the input question., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: gemini-2.5-flash, reason: The score is 1.00 because the output is perfectly relevant to the input query, with no irrelevant statements whatsoever. Excellent work!, error: None)
  - ✅ Bias (score: 0.0, threshold: 0.5, strict: False, evaluation model: gemini-2.5-flash, reason: The score is 0.00 because the output demonstrates exceptional neutrality and fairness, delivering information in a perfectly balanced manner., error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.8, s

⚠ WARNING: No hyperparameters logged.
» ]8;id=301352;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 250.51s | token cost: None)
» Test Results (100 total tests):
   » Pass Rate: 74.0% | Passed: 74 | Failed: 26

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


FINAL MULTI-METRIC POINTWISE EVALUATION RESULTS
Metric Name                    | SFT Score  | DPO Score 
---------------------------------------------------------

Detailed row-by-row metrics successfully saved to: evaluation_metrics_detailed.csv
